##### STA 220 Data & Web Technologies for Data Analysis

### Lecture 13, 02/17/2026, Cartography

#### Announcements

- Second homework assignment due February 20.

### Today's topics

Interactive plots:
- Image processing
- Cartography

# Interactive plots

## Maps

The __folium__ package uses the Leaflet JavaScript library to make interactive maps.

The function to create a map is `folium.Map()`. The function's parameters control the position, style, and initial zoom of the map.

If you want to change the size of the map, you first need to create a `folium.Figure()`, and then add the map to the figure with `.add_child()`.

In [1]:
import folium
import folium.plugins

In [2]:
folium.Map(location = [38.54, -121.75], zoom_start = 15)

In [3]:
m = folium.Map(width = 500, height = 500) # not ideal
m

In [4]:
# Make a map.
m = folium.Map(location = [38.54000, -121.74771], zoom_start = 18)
# Davis: 38.5449, -121.7405

# optional: set up a Figure to control the size of the map
fig = folium.Figure(width = 600, height = 400)
fig.add_child(m)

In [5]:
from IPython.display import display
def show_map(m, w = 800, h = 500):
    fig = folium.Figure(width = w, height = h)
    fig.add_child(m)
    display(m)

We can change the tiles. For more details, see [here](https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html) 

In [6]:
m = folium.Map(tiles = "cartodbpositron") # change tile
show_map(m)

In [16]:
import folium.plugins
m = folium.plugins.DualMap(zoom_start=8)

folium.TileLayer("openstreetmap").add_to(m.m1)
folium.TileLayer("cartodbpositron").add_to(m.m2)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Terrain_Base/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite'
).add_to(m.m2)
folium.LayerControl(collapsed=True).add_to(m)

m

Some Plugins:

In [17]:
m = folium.Map(width = 800, height = 600, location = [0,0], zoom_start = 5)
folium.plugins.Fullscreen(position = 'topleft', force_separate_button=False,).add_to(m) # add the fullscreen button
folium.plugins.Geocoder().add_to(m) # find a location via https://nominatim.org/ LIMITED to 1 request/second!

folium.plugins.LocateControl(auto_start=False).add_to(m) # let your browser find your location
folium.plugins.MiniMap(zoom_level_offset=-7, toggle_display=True).add_to(m) # minimap
folium.plugins.Terminator().add_to(m) # daylight/shadow
m.add_child(
    folium.LatLngPopup() # if you click somewhere, you'll see a Popup with your location
)
show_map(m, 800, 600)

In [18]:
import folium
from folium.plugins import Draw

m = folium.Map()

Draw(export=True).add_to(m)

show_map(m)

The [Yolo County Restuarants Dataset](http://anson.ucdavis.edu/~nulle/yolo_food.feather) contains locations and health inspector scores for all restaurants in Yolo County, California.

Let's use __folium__ to display the restaurants on a map.

In [19]:
import pandas as pd 

food = pd.read_feather("../data/yolo_food.feather")
food.head()

,Address,CityStateZip,FacilityId,FacilityName,LastScore,attachmentId,facility_index,programId,lat,lng,violation_count
0,507 L ST,DAVIS CA 95616,FA0001050,AGGIE LIQUOR,100.0,None,0,PR0000625,38.548803,-121.734964,1.0
1,1638 W CAPITOL AVE A,WEST SACRAMENTO CA 95691,FA0001104,ARIANA FOOD MARKET,100.0,47e30d7a-1ac8-4f4e-9698-a8470105abf2,1,PR0001009,38.580577,-121.529824,40.0
2,940 SACRAMENTO AVE,WEST SACRAMENTO CA 95605,FA0001394,ARTEAGA'S SUPERMARKET INC,100.0,97e57282-4d8f-489c-b824-a7f901131b7d,2,PR0000916,38.590215,-121.525425,36.0
3,966 SACRAMENTO Ave,WEST SACRAMENTO CA 95691,FA0001628,AY! JALISCO TAQUERIA #1,100.0,None,3,PR0022107,38.589293,-121.524593,15.0
4,220 3RD ST,DAVIS CA 95616,FA0001973,ALI BABA RESTAURANT,100.0,21baede1-18be-40df-a479-a86b00c2551f,4,PR0000674,38.543602,-121.746331,34.0


Unlike most of the plotting packages we used before, __folium__ does not automatically handle missing values. So in order to make our map, we first need to remove the missing values from our dataset.

In [20]:
food_cp = food.copy()

In [21]:
food_cp = food_cp[food_cp.lat.notna() & food_cp.lng.notna()]

In [22]:
food_cp.shape

(770, 11)

In [23]:
food.shape

(965, 11)

Now we can make the map. For each restaurant, we have to create a circle and add it to the map.

In [26]:
m = folium.Map(location = [38.5449, -121.7405], zoom_start = 15)

cols = ["FacilityName", "lat", "lng"]
for name, lat, lng in food_cp[cols].itertuples(index = False):
    popup = folium.Popup(name, parse_html = True)
    circle = folium.Circle([float(lat), float(lng)], color = "red", radius = 10, popup = popup)
    m.add_child(circle)

folium.plugins.LocateControl(auto_start=False).add_to(m) # let your browser find your location
folium.plugins.Fullscreen(position = 'topleft', force_separate_button=False,).add_to(m) # add the fullscreen button
folium.plugins.Geocoder().add_to(m) # find a location via https://nominatim.org/ LIMITED to 1 request/second!

fig = folium.Figure(width = 900, height = 600)
fig.add_child(m)
fig.save("../output/yolo_food_map.html")

In [27]:
m

## GEOCODING

The folium pacakage can be very useful in combination with geocoding, that is, getting the coordinates for a specific address.

Read the [Documentation](https://nominatim.org/release-docs/develop/api/Overview/) of Nominatim API.

Only 1 request/second, User-Agent must be specified, cache must be used.

In [28]:
import requests

In [29]:
url = 'https://nominatim.openstreetmap.org/search'
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:145.0) Gecko/20100101 Firefox/145.0'
}

In [30]:
import requests_cache

# Install cache, specifying a cache file name and optional expiration time
requests_cache.install_cache('../data/geocoding2', expire_after=3600)

In [31]:
response = requests.get(url, headers=headers, params={
    'q': 'Vienna'})

In [32]:
response.raise_for_status()

In [33]:
response.text

'<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset=\'utf-8\'>\n  <meta name=\'viewport\' content=\'width=device-width,initial-scale=1\'>\n\n  <title>Nominatim Demo</title>\n\n  <link rel="icon" type="image/png" href="theme/favicon-194x194.png" sizes="194x194">\n  <link rel="icon" type="image/png" href="theme/favicon-32x32.png" sizes="32x32">\n\n  <link rel=\'stylesheet\' href=\'build/bundle.css\'>\n  <link rel=\'stylesheet\' href=\'theme/style.css\'>\n\n  <script src=\'config.defaults.js\'></script>\n  <script src=\'theme/config.theme.js\'></script>\n\n  <script>\n    if (Nominatim_Config.Reverse_Only) {\n      window.location.pathname = window.location.pathname.replace(\'search.html\', \'reverse.html\');\n    }\n  </script>\n  <script defer src=\'build/bundle.js\'></script>\n</head>\n\n<body>\n</body>\n</html>\n'

In [34]:
response = requests.get(url, headers=headers, params={
    'q': 'Chicago',
    'format': 'json'})

In [35]:
response.raise_for_status()

In [36]:
response.text

'[{"place_id":342264163,"licence":"Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright","osm_type":"relation","osm_id":122604,"lat":"41.8755616","lon":"-87.6244212","class":"boundary","type":"administrative","place_rank":16,"importance":0.8016380325327707,"addresstype":"city","name":"Chicago","display_name":"Chicago, South Chicago Township, Cook County, Illinois, United States","boundingbox":["41.6445310","42.0230529","-87.9400876","-87.5240812"]}]'

More direct approach:

In [37]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="my_geocoding_app")

In [38]:
address = "1600 Amphitheatre Parkway, Mountain View, CA"
location = geolocator.geocode(address)

if location:
    print(f"Address: {location.address}")
    print(f"Latitude: {location.latitude}")
    print(f"Longitude: {location.longitude}")
else:
    print("Location not found.")

Address: Google Building 41, 1600, Amphitheatre Parkway, Mountain View, Santa Clara County, California, 94043, United States
Latitude: 37.4224857
Longitude: -122.0855846


In [39]:
import pandas as pd

In [40]:
df = pd.read_csv('../data/Winter.csv')

In [41]:
df

,index,Year,Host_country,Host_city,Country_Name,Country_Code,Gold,Silver,Bronze
0,0,1924,France,Chamonix,United States,USA,1,2,1
1,1,1924,France,Chamonix,Great Britain,GBR,1,1,2
2,2,1924,France,Chamonix,Austria,AUT,2,1,0
3,3,1924,France,Chamonix,Norway,NOR,4,7,6
4,4,1924,France,Chamonix,Finland,FIN,4,4,3
...,...,...,...,...,...,...,...,...,...
404,404,2018,South Korea,Pyeongchang,Slovakia,SVK,1,2,0
405,405,2018,South Korea,Pyeongchang,China,CHN,1,6,2
406,406,2018,South Korea,Pyeongchang,Hungary,HUN,1,0,0
407,407,2018,South Korea,Pyeongchang,Poland,POL,1,0,1


In [42]:
cities = df['Host_city'].unique()

In [45]:
cities

array(['Chamonix', 'St. Moritz', 'Lake Placid', 'Garmisch-Partenkirchen',
       'Oslo', "Cortina d'Ampezzo", 'Squaw Valley', 'Innsbruck',
       'Grenoble', 'Sapporo', 'Sarajevo', 'Calgary', 'Albertville',
       'Lillehammer', 'Nagano', 'Salt Lake City', 'Turin', 'Vancouver',
       'Sochi', 'Pyeongchang'], dtype=object)

In [46]:
wgames = df[['Host_city', 'Year']].drop_duplicates()

In [47]:
wgames

,Host_city,Year
0,Chamonix,1924
10,St. Moritz,1928
22,Lake Placid,1932
32,Garmisch-Partenkirchen,1936
43,St. Moritz,1948
56,Oslo,1952
69,Cortina d'Ampezzo,1956
82,Squaw Valley,1960
96,Innsbruck,1964
110,Grenoble,1968


In [48]:
wgames.set_index('Host_city', inplace=True)

In [49]:
wgames

,Year
Host_city,
Chamonix,1924
St. Moritz,1928
Lake Placid,1932
Garmisch-Partenkirchen,1936
St. Moritz,1948
Oslo,1952
Cortina d'Ampezzo,1956
Squaw Valley,1960
Innsbruck,1964


In [50]:
wgames['lat'] = None
wgames['lon'] = None

In [51]:
wgames

,Year,lat,lon
Host_city,,,
Chamonix,1924,None,None
St. Moritz,1928,None,None
Lake Placid,1932,None,None
Garmisch-Partenkirchen,1936,None,None
St. Moritz,1948,None,None
Oslo,1952,None,None
Cortina d'Ampezzo,1956,None,None
Squaw Valley,1960,None,None
Innsbruck,1964,None,None


In [52]:
import time

In [53]:
for city in wgames.index:
    add = geolocator.geocode(city)
    time.sleep(1)
    wgames.loc[city, 'lat'] = add.latitude
    wgames.loc[city, 'lon'] = add.longitude

In [54]:
wgames

,Year,lat,lon
Host_city,,,
Chamonix,1924,45.92467,6.872751
St. Moritz,1928,46.497896,9.839243
Lake Placid,1932,44.283119,-73.982832
Garmisch-Partenkirchen,1936,47.492374,11.096281
St. Moritz,1948,46.497896,9.839243
Oslo,1952,59.91333,10.73897
Cortina d'Ampezzo,1956,46.538333,12.137351
Squaw Valley,1960,36.70593,-119.200118
Innsbruck,1964,47.26543,11.392769


In [55]:
import folium

In [56]:
m = folium.Map()

In [57]:
for index, row in wgames.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=index + ": " + str(row['Year']),
    ).add_to(m)

In [62]:
show_map(m, 800, 500)

Now, let's do the same for the summer!

In [63]:
summer = pd.read_csv('../data/Summer_Olympic_Medals.csv')

In [64]:
sgames = summer[['Host_city', 'Year']].drop_duplicates()

In [65]:
sgames

,Host_city,Year
0,Athens,1896
11,Paris,1900
32,St. Louis,1904
45,London,1908
64,Stockholm,1912
83,Antwerp,1920
105,Paris,1924
132,Amsterdam,1928
165,Los Angeles,1932
192,Berlin,1936


In [66]:
sgames[sgames['Year'] == 1956]

,Host_city,Year
304,Melbourne/Stockholm,1956


In [67]:
sgames.loc[sgames['Year'] == 1956, 'Host_city']

304    Melbourne/Stockholm
Name: Host_city, dtype: object

In [68]:
sgames.loc[sgames['Year'] == 1956, 'Host_city'] = 'Melbourne'

In [69]:
sgames[sgames['Year'] == 1956]

,Host_city,Year
304,Melbourne,1956


In [70]:
sgames.set_index('Host_city', inplace=True)

In [71]:
sgames

,Year
Host_city,
Athens,1896
Paris,1900
St. Louis,1904
London,1908
Stockholm,1912
Antwerp,1920
Paris,1924
Amsterdam,1928
Los Angeles,1932


In [72]:
sgames['lat'] = None
sgames['lon'] = None

In [73]:
for city in sgames.index:
    add = geolocator.geocode(city)
    time.sleep(1)
    sgames.loc[city, 'lat'] = add.latitude
    sgames.loc[city, 'lon'] = add.longitude

In [74]:
m = folium.Map()

In [75]:
for index, row in sgames.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(index + ": " + str(row['Year']) + " (Summer)", max_width=200),
        icon=folium.Icon(color='orange')
    ).add_to(m)
for index, row in wgames.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(index + ": " + str(row['Year']) + " (Winter)", max_width=200),
        color='darkblue'
    ).add_to(m)

In [76]:
show_map(m, 800, 500)

In [77]:
# dont run
wgames['type'] = 'Winter'
sgames['type'] = 'Summer'
# df = wgames + sgames
df = pd.concat([wgames, sgames])
df

,Year,lat,lon,type
Host_city,,,,
Chamonix,1924,45.92467,6.872751,Winter
St. Moritz,1928,46.497896,9.839243,Winter
Lake Placid,1932,44.283119,-73.982832,Winter
Garmisch-Partenkirchen,1936,47.492374,11.096281,Winter
St. Moritz,1948,46.497896,9.839243,Winter
Oslo,1952,59.91333,10.73897,Winter
Cortina d'Ampezzo,1956,46.538333,12.137351,Winter
Squaw Valley,1960,36.70593,-119.200118,Winter
Innsbruck,1964,47.26543,11.392769,Winter


In [83]:
m = folium.Map(location = [45, 0], zoom_start = 2)
# Davis: 38.5449, -121.7405

fig = folium.Figure(width = 1100, height = 700)
fig.add_child(m)

winter_group = folium.FeatureGroup(name='Winter Games')
summer_group = folium.FeatureGroup(name='Summer Games')

for index, row in sgames.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(index + ": " + str(row['Year']) + " (Summer)", max_width=200),
        icon=folium.Icon(color="orange", icon="sun", prefix="fa")
    ).add_to(summer_group)
for index, row in wgames.iterrows():
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(index + ": " + str(row['Year']) + " (Winter)", max_width=200),
        icon=folium.Icon(color="blue", icon="snowflake", prefix="fa")
    ).add_to(winter_group)

winter_group.add_to(m)
summer_group.add_to(m)
folium.plugins.Fullscreen(position = 'topleft', force_separate_button=False,).add_to(m) # add the fullscreen button

folium.LayerControl(collapsed=False).add_to(m)

show_map(m, 1100, 700)

In [80]:
m.save("../output/olympic_games.html")